In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv('/content/qoute_dataset.csv')

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
df.shape

(3038, 2)

In [ ]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
qoutes  = quotes.str.lower()

In [ ]:
import string
translator = str.maketrans('','',string.punctuation)
qoutes = qoutes.apply(lambda x: x.translate(translator))

In [ ]:
qoutes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size = 10000

Tokenizer = Tokenizer(num_words=vocab_size)
Tokenizer.fit_on_texts(quotes)

In [ ]:
word_index = Tokenizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8213


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [ ]:
from typing import Sequence
Sequence = Tokenizer.texts_to_sequences(quotes)

In [ ]:
for i in range(3):
  print(quotes[i])

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
“It is our choices, Harry, that show what we truly are, far more than our abilities.”
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”


In [ ]:
for i in range(3):
  print(Sequence[i])

[732, 61, 30, 19, 17, 901, 10, 7, 5, 1188, 8, 71, 290, 10, 149, 12, 835, 103, 776, 71, 290, 136]
[977, 7, 71, 902, 388, 9, 431, 21, 19, 479, 14, 299, 51, 55, 71, 2545, 136]
[1384, 14, 53, 203, 700, 3, 80, 15, 37, 36, 7, 30, 300, 92, 7, 5, 1073, 1, 104, 7, 30, 300, 126, 7, 5, 1073, 136]


In [ ]:
X = []
y = []

for seq in Sequence:
  for i in range(1,len(seq)):
    input_seq = seq[:i]
    output_seq = seq[i]
    X.append(input_seq)
    y.append(output_seq)

In [ ]:
len(X)

86549

In [ ]:
len(y)

86549

In [ ]:
max_len = max(len(x) for x in X)
print(max_len)

748


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen =max_len,padding ='pre' )

In [ ]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  732],
       [   0,    0,    0, ...,    0,  732,   61],
       [   0,    0,    0, ...,  732,   61,   30],
       ...,
       [   0,    0,    0, ...,    9,   19, 1333],
       [   0,    0,    0, ...,   19, 1333,    3],
       [   0,    0,    0, ..., 1333,    3,  174]], dtype=int32)

In [ ]:
y = np.array(y)

In [ ]:
X_padded.shape

(86549, 748)

In [ ]:
y.shape

(86549,)

In [ ]:
from tensorflow.keras.utils import to_categorical

y_one_hot = to_categorical(y, num_classes=vocab_size)

In [ ]:
print(y_one_hot.shape)

(86549, 10000)


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM,Dense
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
embedding_dim = 50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim,input_length=max_len)
)

rnn_model.add(SimpleRNN(units= rnn_units))
rnn_model.add(Dense(units=vocab_size,activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
lstm_model = Sequential()

lstm_model.add(
    Embedding(input_dim=vocab_size,output_dim=embedding_dim, input_length = max_len)
    )
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 100
batch_size = 128

In [ ]:
# early_stopping = EarlyStopping(
#     monitor='val_loss',  # Monitor validation loss
#     patience=5,          # Number of epochs with no improvement after which training will be stopped
#     restore_best_weights=True # Restore model weights from the epoch with the best value of the monitored quantity.
# )

# history_rnn = rnn_model.fit(
#     X_padded,
#     y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split = 0.1,
#     callbacks=[early_stopping]
# )

In [ ]:
history_lstm = lstm_model.fit(
    X_padded,
    y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split = 0.1,
    # callbacks=[early_stopping]
)

Epoch 1/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 38s 53ms/step - accuracy: 0.0359 - loss: 7.0644 - val_accuracy: 0.0508 - val_loss: 6.6284
Epoch 2/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.0551 - loss: 6.3001 - val_accuracy: 0.0662 - val_loss: 6.4955
Epoch 3/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.0770 - loss: 6.0367 - val_accuracy: 0.0853 - val_loss: 6.4075
Epoch 4/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 42s 52ms/step - accuracy: 0.0973 - loss: 5.7951 - val_accuracy: 0.0980 - val_loss: 6.3682
Epoch 5/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.1080 - loss: 5.6060 - val_accuracy: 0.1002 - val_loss: 6.3466
Epoch 6/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.1179 - loss: 5.4315 - val_accuracy: 0.1012 - val_loss: 6.3591
Epoch 7/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 32s 52ms/step - accuracy: 0.1246 - loss: 5.2958 - val_accuracy: 0.1061 - val_loss: 6.3762
Epoch 8/100
609/609 ━━━━━━━━━━━━━━━━━━━━ 31s 51ms/step - accuracy: 0.1365 - loss: 5

In [ ]:
lstm_model.save('lstm_model.h5')

In [ ]:
index_to_word = {}
for word,index in word_index.items():
  index_to_word[index] = word

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
from re import VERBOSE
def predictor(model,Tokenizer,text,max_len):
  text = text.lower()

  seq = Tokenizer.texts_to_sequences([text])[0]
  seq = model.predict([seq], maxlen = max_len,padding = 'pre')


  pred = model.predict(seq,VERBOSE = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [ ]:
seed_text = 'what are you'
next_word = predictor(lstm_model,Tokenizer,seed_text,max_len)
print(next_word)

learn


In [ ]:
def generate_text(model,Tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,Tokenizer,seed_text,max_len)
    if next_word == '':
      break
    seed_text+=" " + next_word
  return seed_text

In [ ]:
seed = 'the meaning of life'
generate_text = generate_text(lstm_model,Tokenizer,seed,max_len,10)
print(generate_text)

the meaning of life is not what it is to be a man to


In [ ]:
import pickle
with open('tokenizer.pkl','wb') as f:
  pickle.dump(Tokenizer,f)

In [ ]:
with open('max_len.pkl','wb') as f:
  pickle.dump(max_len,f)